# Additional Merges: Food × Gut Microbiome Analysis
## Research Question
**How does exposure to food-associated microbes influence the composition and stability of the human gut microbiome?**

### This notebook contains 4 additional merge strategies:

| Merge | Question answered |
|-------|-------------------|
| **1 – Shared taxa matrix** | Which food-associated microbes also appear in the gut? |
| **2 – Temporal FME × gut stability** | Does cumulative food microbe exposure predict gut stability over time? |
| **3 – Fermented vs. non-fermented** | Does fermented food exposure drive different gut outcomes than non-fermented? |
| **4 – FME quartile × beta diversity** | Do high-FME participants cluster differently in PCA space? |

> **Prerequisite:** Run `Central_Merge.ipynb` first and ensure `central_merged.csv` and `fme_sample_scores.csv` exist in the working directory.


---
## Section 1 – Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import mannwhitneyu, kruskal
from scipy.spatial.distance import braycurtis

pd.set_option("display.max_columns", 25)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100
print("Libraries loaded.")


In [ ]:
# ── Load all source files ──────────────────────────────────────────────────────
gut_microbiome   = pd.read_csv("microbiome_filtered.csv", index_col=0)
gut_micro_clr    = pd.read_csv("microbiome_clr.csv",      index_col=0)
sample_metadata  = pd.read_csv("sample_meta.csv",         index_col=0)
cfmd_metadata    = pd.read_csv("cfmd_metadata.csv")
cfmd_abundance   = pd.read_csv("combined_datasets.csv",   index_col=0, low_memory=False)
cfmd_abundance   = cfmd_abundance.apply(pd.to_numeric, errors="coerce").fillna(0.0)
diet_consumption = pd.read_csv("diet_raw.csv",            index_col=0)

# ── Load outputs from Central_Merge.ipynb ─────────────────────────────────────
central_merged   = pd.read_csv("central_merged.csv",      index_col=0)
fme_scores       = pd.read_csv("fme_sample_scores.csv",   index_col=0)

# Core sample set (has both gut microbiome and diet record)
SAMPLE_COL_PREFIX = "MCT.f."
core_sample_ids   = sorted(
    set(gut_microbiome.columns) &
    set(c for c in diet_consumption.columns if c.startswith(SAMPLE_COL_PREFIX))
)

print("Files loaded:")
for name, df in [
    ("gut_microbiome",  gut_microbiome),
    ("gut_micro_clr",   gut_micro_clr),
    ("cfmd_abundance",  cfmd_abundance),
    ("central_merged",  central_merged),
    ("fme_scores",      fme_scores),
]:
    print(f"  {name:<20} {str(df.shape)}")
print(f"\nCore analytical samples: {len(core_sample_ids)}")


---
## Merge 1 – Shared Taxa Matrix
### Question: Which food-associated microbes also appear in the gut?

**Method:**
- Parse species-level names from both gut (`microbiome_filtered`) and food (`combined_datasets`) taxonomy strings
- Find the intersection (shared species)
- Build a matrix: shared taxa × all samples (gut + food), enabling direct abundance comparison

**Why it matters:** Shared taxa are the candidates for food-to-gut microbial transmission — the biological mechanism underlying the research question.


In [ ]:
# ── Taxonomy parsers ──────────────────────────────────────────────────────────
def clean_gut_species(tax_string):
    """Extract clean species/genus name from semicolon-delimited gut taxonomy."""
    parts = str(tax_string).split(";")
    for part in reversed(parts):
        part = part.strip()
        val  = part.replace("s__","").replace("g__","").replace("f__","").strip()
        if val and val != "NA" and not val.startswith("k__") and not val.startswith("p__"):
            return val.lower().replace(" ", "_")
    return None

def clean_food_species(tax_string):
    """Extract clean species/genus name from pipe-delimited cFMD taxonomy (skip t__ strain level)."""
    parts = str(tax_string).split("|")
    for part in reversed(parts):
        part = part.strip()
        if part.startswith("t__"):
            continue
        val = part.replace("s__","").replace("g__","").strip()
        if val and not val.startswith("k__") and not val.startswith("p__"):
            return val.lower().replace(" ", "_")
    return None

# ── Build species maps for both datasets ──────────────────────────────────────
gut_species_map  = {t: clean_gut_species(t)  for t in gut_microbiome.index}
food_species_map = {t: clean_food_species(t) for t in cfmd_abundance.index}

gut_species_set  = set(v for v in gut_species_map.values()  if v)
food_species_set = set(v for v in food_species_map.values() if v)
shared_species   = gut_species_set & food_species_set

print(f"Gut unique species   : {len(gut_species_set)}")
print(f"Food unique species  : {len(food_species_set)}")
print(f"Shared species       : {len(shared_species)}")
print(f"Overlap %            : {100*len(shared_species)/len(gut_species_set):.1f}% of gut taxa found in food")
print()
print("Top shared species (first 20):")
for s in sorted(shared_species)[:20]:
    print(f"  {s}")


In [ ]:
# ── Build shared taxa abundance matrix ────────────────────────────────────────

# Gut: reindex by clean species, keep core samples only
gut_for_shared = gut_microbiome[core_sample_ids].copy()
gut_for_shared.index = gut_for_shared.index.map(gut_species_map)
gut_for_shared = (
    gut_for_shared[gut_for_shared.index.isin(shared_species)]
    .groupby(level=0).mean()
)

# Food: reindex by clean species
food_for_shared = cfmd_abundance.copy()
food_for_shared.index = food_for_shared.index.map(food_species_map)
food_for_shared = (
    food_for_shared[food_for_shared.index.isin(shared_species)]
    .groupby(level=0).mean()
)

# Rename columns to distinguish sources
gut_for_shared.columns  = ["gut__"  + c for c in gut_for_shared.columns]
food_for_shared.columns = ["food__" + c for c in food_for_shared.columns]

# Merge on shared taxa
shared_taxa_matrix = gut_for_shared.join(food_for_shared, how="inner")
shared_taxa_matrix.index.name = "species"

print(f"Shared taxa matrix: {shared_taxa_matrix.shape}")
print(f"  Shared taxa (rows) : {shared_taxa_matrix.shape[0]}")
print(f"  Gut columns        : {gut_for_shared.shape[1]}")
print(f"  Food columns       : {food_for_shared.shape[1]}")

shared_taxa_matrix.to_csv("shared_taxa_matrix.csv")
print("\nSaved: shared_taxa_matrix.csv")


In [ ]:
# ── Visualize: mean gut vs. mean food abundance for shared taxa ───────────────
gut_cols  = [c for c in shared_taxa_matrix.columns if c.startswith("gut__")]
food_cols = [c for c in shared_taxa_matrix.columns if c.startswith("food__")]

mean_gut_abund  = shared_taxa_matrix[gut_cols].mean(axis=1)
mean_food_abund = shared_taxa_matrix[food_cols].mean(axis=1)

comparison_df = pd.DataFrame({
    "mean_gut_abundance" : mean_gut_abund,
    "mean_food_abundance": mean_food_abund,
    "species"            : shared_taxa_matrix.index,
}).sort_values("mean_gut_abundance", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top 20 shared taxa by gut abundance
top20 = comparison_df.head(20)
axes[0].barh(top20["species"][::-1], top20["mean_gut_abundance"][::-1],
             color="steelblue", edgecolor="none")
axes[0].set_title("Top 20 Shared Taxa\n(by mean gut abundance)")
axes[0].set_xlabel("Mean Relative Abundance (gut)")

# Scatter: gut abundance vs. food abundance
axes[1].scatter(comparison_df["mean_food_abundance"],
                comparison_df["mean_gut_abundance"],
                alpha=0.6, s=30, c="teal", edgecolors="none")
axes[1].set_xlabel("Mean Abundance in Food (cFMD)")
axes[1].set_ylabel("Mean Abundance in Gut")
axes[1].set_title("Shared Taxa:\nFood Abundance vs. Gut Abundance")

plt.suptitle("Merge 1: Shared Taxa — Food × Gut Microbiome", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


---
## Merge 2 – Temporal FME × Gut Stability
### Question: Does cumulative food microbe exposure predict gut stability over time?

**Method:**
- For each participant, align FME scores with gut microbiome samples along the study timeline (days 1–17)
- Compute rolling 3-day cumulative FME exposure
- Compute gut microbiome stability as sample-to-sample Bray-Curtis dissimilarity between consecutive time points
- Merge into a longitudinal table: one row per consecutive sample pair

**Why it matters:** If food microbe exposure stabilizes the gut microbiome, we expect lower gut dissimilarity on days following high-FME dietary periods.


In [ ]:
# ── Build longitudinal table: sample × study_day × participant ────────────────
longitudinal_df = (
    central_merged[["participant_id", "StudyDayNo",
                    "fme_total", "fme_fermented", "fme_nonfermented",
                    "shannon_diversity", "KCAL", "FIBE"]]
    .dropna(subset=["participant_id", "StudyDayNo", "fme_total"])
    .copy()
)
longitudinal_df["StudyDayNo"] = pd.to_numeric(longitudinal_df["StudyDayNo"], errors="coerce")
longitudinal_df = longitudinal_df.sort_values(["participant_id", "StudyDayNo"])
longitudinal_df.index.name = "sample_id"
longitudinal_df = longitudinal_df.reset_index()

print(f"Longitudinal table: {longitudinal_df.shape}")
display(longitudinal_df.head(6))


In [ ]:
# ── Compute rolling 3-day cumulative FME per participant ──────────────────────
longitudinal_df["fme_rolling3"] = (
    longitudinal_df
    .groupby("participant_id")["fme_total"]
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
)

# ── Compute Bray-Curtis dissimilarity between consecutive gut samples ──────────
gut_norm = gut_microbiome[core_sample_ids].apply(
    lambda col: col / col.sum() if col.sum() > 0 else col, axis=0
)

bc_records = []
for participant, group in longitudinal_df.groupby("participant_id"):
    group = group.sort_values("StudyDayNo")
    sample_ids = group["sample_id"].tolist()

    for i in range(len(sample_ids) - 1):
        s1, s2 = sample_ids[i], sample_ids[i+1]
        if s1 in gut_norm.columns and s2 in gut_norm.columns:
            v1 = gut_norm[s1].values
            v2 = gut_norm[s2].values
            bc = braycurtis(v1, v2)
            day1 = group.loc[group["sample_id"] == s1, "StudyDayNo"].values[0]
            day2 = group.loc[group["sample_id"] == s2, "StudyDayNo"].values[0]
            bc_records.append({
                "participant_id"   : participant,
                "sample_id_t1"     : s1,
                "sample_id_t2"     : s2,
                "study_day_t1"     : day1,
                "study_day_t2"     : day2,
                "bray_curtis_dissim": bc,
            })

bc_df = pd.DataFrame(bc_records)
print(f"Consecutive sample pairs with BC dissimilarity: {len(bc_df)}")
display(bc_df.head(5))


In [ ]:
# ── Merge BC dissimilarity with FME at t1 ────────────────────────────────────
temporal_merge = bc_df.merge(
    longitudinal_df[["sample_id","participant_id","StudyDayNo",
                     "fme_total","fme_rolling3","fme_fermented",
                     "fme_nonfermented","shannon_diversity","KCAL","FIBE"]]
    .rename(columns={
        "sample_id"       : "sample_id_t1",
        "StudyDayNo"      : "study_day_t1_check",
        "shannon_diversity": "shannon_t1",
        "fme_total"       : "fme_at_t1",
        "fme_rolling3"    : "fme_rolling3_at_t1",
        "fme_fermented"   : "fme_ferm_at_t1",
        "fme_nonfermented": "fme_nonferm_at_t1",
    }),
    on=["sample_id_t1","participant_id"],
    how="left"
)

print(f"Temporal merge table: {temporal_merge.shape}")
print(f"BC dissim range: {temporal_merge['bray_curtis_dissim'].min():.3f} – {temporal_merge['bray_curtis_dissim'].max():.3f}")

temporal_merge.to_csv("temporal_fme_stability.csv", index=False)
print("Saved: temporal_fme_stability.csv")


In [ ]:
# ── Visualize: FME (rolling 3-day) vs Bray-Curtis dissimilarity ──────────────
plot_df = temporal_merge.dropna(subset=["fme_rolling3_at_t1","bray_curtis_dissim"])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: rolling FME vs next-day gut dissimilarity
axes[0].scatter(plot_df["fme_rolling3_at_t1"], plot_df["bray_curtis_dissim"],
                alpha=0.4, s=25, c="darkorange", edgecolors="none")
if len(plot_df) > 2:
    z = np.polyfit(plot_df["fme_rolling3_at_t1"], plot_df["bray_curtis_dissim"], 1)
    x_line = np.linspace(plot_df["fme_rolling3_at_t1"].min(),
                         plot_df["fme_rolling3_at_t1"].max(), 100)
    axes[0].plot(x_line, np.poly1d(z)(x_line), "r--", lw=1.5, label="Trend")
axes[0].set_title("Rolling 3-day FME vs. Gut Dissimilarity\n(consecutive sample pairs)")
axes[0].set_xlabel("Rolling 3-day Mean FME Score")
axes[0].set_ylabel("Bray-Curtis Dissimilarity (t → t+1)")
axes[0].legend()

# Timeline: mean FME and mean BC dissimilarity by study day
day_summary = (
    temporal_merge
    .groupby("study_day_t1")
    .agg(mean_fme=("fme_at_t1","mean"), mean_bc=("bray_curtis_dissim","mean"))
    .dropna()
)
ax2 = axes[1]
ax2b = ax2.twinx()
ax2.bar(day_summary.index, day_summary["mean_fme"], color="steelblue",
        alpha=0.5, label="Mean FME")
ax2b.plot(day_summary.index, day_summary["mean_bc"], color="red",
          marker="o", markersize=4, linewidth=1.5, label="Mean BC dissim")
ax2.set_xlabel("Study Day")
ax2.set_ylabel("Mean FME Score", color="steelblue")
ax2b.set_ylabel("Mean Bray-Curtis Dissimilarity", color="red")
ax2.set_title("FME and Gut Instability Over Study Timeline")
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax2.legend(lines1+lines2, labels1+labels2, loc="upper right", fontsize=9)

plt.suptitle("Merge 2: Temporal FME × Gut Stability", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


---
## Merge 3 – Fermented vs. Non-Fermented Food Exposure
### Question: Does fermented food drive different gut outcomes than non-fermented food?

**Method:**
- Split FME scores by fermentation status using the `fermented/non-fermented` column in cFMD metadata
- Build a participant-level table with separate FME scores for fermented (F) and non-fermented (NF) foods
- Compare gut alpha diversity and stability between participants with high fermented-FME vs. high non-fermented-FME

**Why it matters:** Fermented foods carry live microbes that may actively colonize the gut, while non-fermented foods carry environmental microbes. The biological pathway differs — and the effect on the gut microbiome may differ too.


In [ ]:
# ── Build participant-level fermented/non-fermented FME summary ──────────────
ferm_merge = central_merged[
    ["participant_id", "StudyDayNo",
     "fme_total", "fme_fermented", "fme_nonfermented",
     "shannon_diversity", "Age", "BMI", "Gender",
     "KCAL", "FIBE", "D_YOGURT", "D_CHEESE"]
].dropna(subset=["participant_id","fme_fermented","fme_nonfermented"]).copy()

# Participant-level aggregation
participant_ferm = (
    ferm_merge.groupby("participant_id")
    .agg(
        fme_total_mean       = ("fme_total",         "mean"),
        fme_fermented_mean   = ("fme_fermented",      "mean"),
        fme_nonfermented_mean= ("fme_nonfermented",   "mean"),
        shannon_mean         = ("shannon_diversity",   "mean"),
        shannon_std          = ("shannon_diversity",   "std"),
        Age                  = ("Age",                "first"),
        BMI                  = ("BMI",                "first"),
        Gender               = ("Gender",             "first"),
        KCAL_mean            = ("KCAL",               "mean"),
        FIBE_mean            = ("FIBE",               "mean"),
        n_samples            = ("fme_total",          "count"),
    )
    .reset_index()
)

# Stability = CV of shannon diversity
participant_ferm["shannon_cv"] = (
    participant_ferm["shannon_std"] / participant_ferm["shannon_mean"]
)

# Fermented FME fraction
participant_ferm["fme_ferm_fraction"] = (
    participant_ferm["fme_fermented_mean"] /
    (participant_ferm["fme_total_mean"] + 1e-9)
)

print(f"Participants in fermented merge: {len(participant_ferm)}")
display(participant_ferm.head(5))


In [ ]:
# ── Split participants into high/low fermented FME groups ─────────────────────
ferm_median = participant_ferm["fme_fermented_mean"].median()
nonferm_median = participant_ferm["fme_nonfermented_mean"].median()

participant_ferm["fme_ferm_group"]    = np.where(
    participant_ferm["fme_fermented_mean"]    >= ferm_median,    "high_fermented", "low_fermented")
participant_ferm["fme_nonferm_group"] = np.where(
    participant_ferm["fme_nonfermented_mean"] >= nonferm_median, "high_nonfermented", "low_nonfermented")

# Statistical test: Mann-Whitney U
high_f  = participant_ferm.loc[participant_ferm["fme_ferm_group"]=="high_fermented",  "shannon_mean"].dropna()
low_f   = participant_ferm.loc[participant_ferm["fme_ferm_group"]=="low_fermented",   "shannon_mean"].dropna()
high_nf = participant_ferm.loc[participant_ferm["fme_nonferm_group"]=="high_nonfermented","shannon_mean"].dropna()
low_nf  = participant_ferm.loc[participant_ferm["fme_nonferm_group"]=="low_nonfermented", "shannon_mean"].dropna()

stat_f,  p_f  = mannwhitneyu(high_f,  low_f,  alternative="two-sided") if len(high_f)>1  and len(low_f)>1  else (np.nan, np.nan)
stat_nf, p_nf = mannwhitneyu(high_nf, low_nf, alternative="two-sided") if len(high_nf)>1 and len(low_nf)>1 else (np.nan, np.nan)

print(f"Fermented FME   — high vs. low → Shannon diversity: U={stat_f:.1f}, p={p_f:.4f}")
print(f"Non-fermented FME — high vs. low → Shannon diversity: U={stat_nf:.1f}, p={p_nf:.4f}")

participant_ferm.to_csv("fermented_nonfermented_merge.csv", index=False)
print("\nSaved: fermented_nonfermented_merge.csv")


In [ ]:
# ── Visualize: fermented vs. non-fermented FME impact on gut diversity ────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

palette = {"high_fermented":"#2ecc71","low_fermented":"#95a5a6",
           "high_nonfermented":"#e74c3c","low_nonfermented":"#bdc3c7"}

# Boxplot: fermented FME group vs Shannon
groups_f = [
    participant_ferm.loc[participant_ferm["fme_ferm_group"]=="high_fermented", "shannon_mean"].dropna(),
    participant_ferm.loc[participant_ferm["fme_ferm_group"]=="low_fermented",  "shannon_mean"].dropna(),
]
axes[0].boxplot(groups_f, labels=["High fermented FME","Low fermented FME"], patch_artist=True,
                boxprops=dict(facecolor="#2ecc71", alpha=0.6))
axes[0].set_title(f"Fermented FME vs. Gut Diversity\n(p={p_f:.3f})")
axes[0].set_ylabel("Mean Shannon Diversity")

# Boxplot: non-fermented FME group vs Shannon
groups_nf = [
    participant_ferm.loc[participant_ferm["fme_nonferm_group"]=="high_nonfermented","shannon_mean"].dropna(),
    participant_ferm.loc[participant_ferm["fme_nonferm_group"]=="low_nonfermented", "shannon_mean"].dropna(),
]
axes[1].boxplot(groups_nf, labels=["High non-ferm FME","Low non-ferm FME"], patch_artist=True,
                boxprops=dict(facecolor="#e74c3c", alpha=0.6))
axes[1].set_title(f"Non-fermented FME vs. Gut Diversity\n(p={p_nf:.3f})")
axes[1].set_ylabel("Mean Shannon Diversity")

# Scatter: fermented FME vs non-fermented FME (coloured by shannon)
sc = axes[2].scatter(
    participant_ferm["fme_fermented_mean"],
    participant_ferm["fme_nonfermented_mean"],
    c=participant_ferm["shannon_mean"], cmap="RdYlGn", s=80,
    edgecolors="white", linewidths=0.5, vmin=2, vmax=4
)
plt.colorbar(sc, ax=axes[2], label="Mean Shannon Diversity")
axes[2].set_xlabel("Mean Fermented FME")
axes[2].set_ylabel("Mean Non-Fermented FME")
axes[2].set_title("Fermented vs. Non-Fermented FME\n(colour = gut diversity)")

plt.suptitle("Merge 3: Fermented vs. Non-Fermented Food Exposure", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


---
## Merge 4 – FME Quartile × Beta Diversity (PCA)
### Question: Do high-FME participants cluster differently in microbiome composition space?

**Method:**
- Assign each sample to an FME quartile (Q1=lowest, Q4=highest food microbe exposure)
- Run PCA on CLR-transformed gut microbiome profiles (beta diversity)
- Plot samples colored by FME quartile
- Test whether FME quartile groups separate in PCA space

**Why it matters:** If food microbe exposure shapes gut microbiome composition, high-FME and low-FME samples should occupy different regions of the composition space.


In [ ]:
# ── Assign FME quartiles to core samples ──────────────────────────────────────
fme_quartile_df = (
    central_merged[["participant_id","fme_total","StudyDayNo"]]
    .dropna(subset=["fme_total"])
    .copy()
)
fme_quartile_df.index.name = "sample_id"
fme_quartile_df = fme_quartile_df.reset_index()

fme_quartile_df["fme_quartile"] = pd.qcut(
    fme_quartile_df["fme_total"], q=4,
    labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"]
)

print("FME quartile distribution:")
print(fme_quartile_df["fme_quartile"].value_counts().sort_index())
print()
print(f"FME quartile thresholds:")
print(pd.qcut(fme_quartile_df["fme_total"], q=4).unique().sort_values())


In [ ]:
# ── Run PCA on CLR-transformed gut microbiome ─────────────────────────────────
# Subset CLR matrix to samples that have FME quartile assigned
pca_samples = [s for s in fme_quartile_df["sample_id"].tolist()
               if s in gut_micro_clr.columns]

clr_matrix = (
    gut_micro_clr[pca_samples]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0.0)
    .T   # samples as rows for PCA
)

# PCA
pca = PCA(n_components=2)
pca_coords = pca.fit_transform(clr_matrix)
explained  = pca.explained_variance_ratio_ * 100

pca_df = pd.DataFrame({
    "sample_id": pca_samples,
    "PC1"      : pca_coords[:, 0],
    "PC2"      : pca_coords[:, 1],
})

# Merge with FME quartile labels
pca_df = pca_df.merge(
    fme_quartile_df[["sample_id","fme_quartile","fme_total","participant_id"]],
    on="sample_id", how="left"
)

print(f"PCA computed for {len(pca_df)} samples")
print(f"PC1: {explained[0]:.1f}%  |  PC2: {explained[1]:.1f}%")

pca_df.to_csv("fme_quartile_pca.csv", index=False)
print("Saved: fme_quartile_pca.csv")


In [ ]:
# ── Visualize PCA coloured by FME quartile ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

quartile_colors = {
    "Q1 (lowest)" : "#3498db",
    "Q2"          : "#2ecc71",
    "Q3"          : "#f39c12",
    "Q4 (highest)": "#e74c3c",
}

# PCA scatter coloured by FME quartile
for quartile, color in quartile_colors.items():
    mask = pca_df["fme_quartile"] == quartile
    axes[0].scatter(
        pca_df.loc[mask, "PC1"], pca_df.loc[mask, "PC2"],
        c=color, label=quartile, alpha=0.7, s=40, edgecolors="white", linewidths=0.3
    )
axes[0].set_xlabel(f"PC1 ({explained[0]:.1f}% variance)")
axes[0].set_ylabel(f"PC2 ({explained[1]:.1f}% variance)")
axes[0].set_title("Beta Diversity PCA — Colored by FME Quartile")
axes[0].legend(title="FME Quartile", fontsize=9)

# Boxplot: PC1 score by FME quartile
quartile_order = ["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"]
pc1_by_quartile = [
    pca_df.loc[pca_df["fme_quartile"] == q, "PC1"].dropna().values
    for q in quartile_order
]
bp = axes[1].boxplot(pc1_by_quartile, labels=quartile_order, patch_artist=True,
                     medianprops=dict(color="black", linewidth=2))
for patch, color in zip(bp["boxes"], quartile_colors.values()):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title("PC1 Score by FME Quartile\n(PC1 captures main composition axis)")
axes[1].set_xlabel("FME Quartile")
axes[1].set_ylabel("PC1 Score")
axes[1].tick_params(axis='x', rotation=20)

# Kruskal-Wallis test across quartiles
valid_groups = [g for g in pc1_by_quartile if len(g) > 1]
if len(valid_groups) >= 2:
    stat_kw, p_kw = kruskal(*valid_groups)
    axes[1].text(0.98, 0.02, f"Kruskal-Wallis p={p_kw:.4f}",
                 transform=axes[1].transAxes, ha="right", fontsize=9,
                 color="darkred" if p_kw < 0.05 else "gray")

plt.suptitle("Merge 4: FME Quartile × Beta Diversity (PCA)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


---
## Summary of All Merges

| Merge | Output file | Key variable | Used for |
|-------|-------------|--------------|----------|
| 1 – Shared taxa | `shared_taxa_matrix.csv` | taxa × (gut+food) samples | Identifying candidate transmission species |
| 2 – Temporal | `temporal_fme_stability.csv` | consecutive sample pairs | Longitudinal FME → gut stability model |
| 3 – Fermented | `fermented_nonfermented_merge.csv` | participant-level | Comparing fermented vs. non-fermented effects |
| 4 – PCA | `fme_quartile_pca.csv` | sample-level PCA coords | Beta diversity × FME exposure groups |

### Recommended statistical models (next notebook)
- **Merge 1** → Fisher's exact test / hypergeometric test on shared taxa prevalence
- **Merge 2** → Linear mixed-effects: `BC_dissim ~ fme_rolling3 + (1|participant_id)`
- **Merge 3** → Mann-Whitney U / linear model: `shannon ~ fme_fermented + fme_nonfermented + covariates`
- **Merge 4** → PERMANOVA on full CLR matrix with FME quartile as grouping variable
